In [ ]:
%reload_ext autoreload
%autoreload 2

from comet_ml import Experiment
import h5py
import matplotlib.pyplot as plt
import numpy as np
import argparse


import importlib
import random
import os
from algorithms.server.server import Server
from algorithms.trainmodel.models import *
from utils.plot_utils import *
import torch

#torch.set_default_dtype(torch.float64)


def main(experiment, dataset, algorithm, model, batch_size, learning_rate, alpha, eta, L, rho, num_glob_iters,
         local_epochs, optimizer, numedges, times, commet, gpu):

    device = torch.device("cuda:{}".format(gpu) if torch.cuda.is_available() and gpu != -1 else "cpu")

    for i in range(times):
        print("---------------Running time:------------",i)

        # Generate model
        if(model == "mclr"):
            if(dataset == "human_activity"):
                model = Mclr_CrossEntropy(561,6).to(device), model
                print("Hey")
            else:
                model = Mclr_Logistic().to(device), model

        if(model == "linear_regression"):
            model = Linear_Regression(40,1).to(device), model

        if model == "logistic_regression":
            model = Logistic_Regression(300).to(device), model
        
        if model == "MLP" and dataset == "a9a":
            model = DNN( input_dim = 123, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "human_activity":
            model = DNN( input_dim = 561, mid_dim = 561, output_dim = 6).to(device), model
        if model == "MLP" and dataset == "w8a":
            model = DNN( input_dim = 300, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "Mnist":
            model = DNN().to(device), model
        if model == 'CNN':
            model = Net().to(device), model
        # select algorithm
        if(commet):
            experiment.set_name(dataset + "_" + algorithm + "_" + model[1] + "_" + str(batch_size) + "b_" + str(learning_rate) + "lr_" + str(alpha) + "al_" + str(eta) + "eta_" + str(L) + "L_" + str(rho) + "p_" +  str(num_glob_iters) + "ge_"+ str(local_epochs) + "le_"+ str(numedges) +"u")
        server = Server(experiment, device, dataset, algorithm, model, batch_size, learning_rate, alpha, eta,  L, num_glob_iters, local_epochs, optimizer, numedges, i)
        
        server.train()
        server.test()


# Training for the MNIST MLP

In [ ]:
sophiaOTA_params = {
    "dataset": "Mnist",
    "algorithm": "Sophia_OTA",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 20.,
    "L": 0.001,
    "rho": 200,
    "num_glob_iters": 2 * 1280,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 32,
    "times": 1,
    "commet": False,
    "gpu": 0
}

sophia_params = {
    "dataset": "Mnist",
    "algorithm": "Sophia",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 20.,
    "L": 0.001,
    "rho": 200,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 32,
    "times": 1,
    "commet": False,
    "gpu": 0
}



DONE_params = {
    "dataset": "Mnist",
    "algorithm": "DONE",
    "model": "MLP",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.003,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 41,
    "local_epochs": 40,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 1,
    "commet": True,
    "gpu": 0
}

GD_params = {
    "dataset": "Mnist",
    "algorithm": "GD",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": 0.03,
    "eta": 20.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "DONE",
    "numedges": 30,
    "times": 1,
    "commet": False,
    "gpu": 0
}

'''
Newton_params = {
    "dataset": "Mnist",
    "algorithm": "Newton",
    "model": "MLP",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.03,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 500,
    "local_epochs": 40,
    "optimizer": "Newton",
    "numedges": 30,
    "times": 1,
    "commet": True,
    "gpu": 0
}

'''

In [ ]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **sophiaOTA_params)
experiment.end()    


In [ ]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **sophia_params)
experiment.end()    

In [ ]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **GD_params)
experiment.end()    

In [ ]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **DONE_params)
experiment.end()    

# Training for the CNN

In [1]:
%reload_ext autoreload
%autoreload 2

from comet_ml import Experiment
import h5py
import matplotlib.pyplot as plt
import numpy as np
import argparse


import importlib
import random
import os
from algorithms.server.server import Server
from algorithms.trainmodel.models import *
from utils.plot_utils import *
import torch
#torch.manual_seed(0)

def main(experiment, dataset, algorithm, model, batch_size, learning_rate, alpha, eta, L, rho, num_glob_iters,
         local_epochs, optimizer, numedges, times, commet, gpu):

    device = torch.device("cuda:{}".format(gpu) if torch.cuda.is_available() and gpu != -1 else "cpu")

    for i in range(times):
        print("---------------Running time:------------",i)

        # Generate model
        if(model == "mclr"):
            if(dataset == "human_activity"):
                model = Mclr_CrossEntropy(561,6).to(device), model
                print("Hey")
            else:
                model = Mclr_Logistic().to(device), model

        if(model == "linear_regression"):
            model = Linear_Regression(40,1).to(device), model

        if model == "logistic_regression":
            model = Logistic_Regression(300).to(device), model
        
        if model == "MLP" and dataset == "a9a":
            model = DNN( input_dim = 123, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "human_activity":
            model = DNN( input_dim = 561, mid_dim = 561, output_dim = 6).to(device), model
        if model == "MLP" and dataset == "w8a":
            model = DNN( input_dim = 300, output_dim = 2).to(device), model
        if model == "MLP" and dataset == "Mnist":
            model = DNN().to(device), model
        if model == 'CNN':
            model = Net().to(device), model
        if model == 'CNN2':
            model = Net3().to(device), model
            
        # select algorithm
        if(commet):
            experiment.set_name(dataset + "_" + algorithm + "_" + model[1] + "_" + str(batch_size) + "b_" + str(learning_rate) + "lr_" + str(alpha) + "al_" + str(eta) + "eta_" + str(L) + "L_" + str(rho) + "p_" +  str(num_glob_iters) + "ge_"+ str(local_epochs) + "le_"+ str(numedges) +"u")
        server = Server(experiment, device, dataset, algorithm, model, batch_size, learning_rate, alpha, eta,  L, num_glob_iters, local_epochs, optimizer, numedges, i)
        
        server.train()
        server.test()

        
sophia_params = {
    "dataset": "Cifar10",
    "algorithm": "Sophia",
    "model": "CNN2",
    "batch_size": 64,
    "learning_rate": 0.0001,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 30.,
    "L": 0.0001,
    "rho": 20,
    "num_glob_iters": 300,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 20,
    "times": 1,
    "commet": False,
    "gpu": 0
}

sophiaOTA_params = {
    "dataset": "Cifar10",
    "algorithm": "Sophia_OTA",
    "model": "CNN",
    "batch_size": 64,
    "learning_rate": 0.0001,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 20,
    "L": 0.001,
    "rho": 20,
    "num_glob_iters": 32 * 300,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 20,
    "times": 5,
    "commet": False,
    "gpu": 0
}



DONE_params = {
    "dataset": "Fashion_Mnist",
    "algorithm": "DONE",
    "model": "CNN",
    "batch_size": 0,
    "learning_rate": 1,
    "alpha": 0.003,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 2 * 41,
    "local_epochs": 40,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 1,
    "commet": False,
    "gpu": 0
}

GD_params = {
    "dataset": "Cifar10",
    "algorithm": "FedAvg",
    "model": "CNN2",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": 0.03,
    "eta": 20.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 300,
    "local_epochs": 10,
    "optimizer": "DONE",
    "numedges": 20,
    "times": 1,
    "commet": False,
    "gpu": 0
}

In [2]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **sophia_params)
experiment.end()  

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/abdulmomen96/sophia/e1aa7781b43d4ab59bb3593c3852e0cb



---------------Running time:------------ 0


COMET WARNING: Failed to log system metrics: [sys.ram,sys.cpu,sys.load]


Number of edges / total edges: 20  /  20
Total number of parameters: 319178
-------------Round number:  0  -------------
Win rate = 0.01865103484576003
-------------Round number:  1  -------------
Average Test Accuracy          :  0.0981349020656461
Average Global Trainning Accuracy:  0.09850832794488171
Average Global Trainning Loss    :  2.304447458416018
Win rate = 0.0545588981696733
-------------Round number:  2  -------------
Average Test Accuracy          :  0.10736011765492345
Average Global Trainning Accuracy:  0.1085419964770675
Average Global Trainning Loss    :  2.301372382884791
Win rate = 0.055658598023673345
-------------Round number:  3  -------------
Average Test Accuracy          :  0.12607794638679057
Average Global Trainning Accuracy:  0.12381546968717251
Average Global Trainning Loss    :  2.2982548036188097
Win rate = 0.05790186040391254
-------------Round number:  4  -------------
Average Test Accuracy          :  0.12975466274483588
Average Global Trainning Accur

Win rate = 0.5047309025058118
-------------Round number:  35  -------------
Average Test Accuracy          :  0.2863827795975667
Average Global Trainning Accuracy:  0.2846217306963366
Average Global Trainning Loss    :  2.046618061439497
Win rate = 0.509844036869709
-------------Round number:  36  -------------
Average Test Accuracy          :  0.28798716491744103
Average Global Trainning Accuracy:  0.2863386028673995
Average Global Trainning Loss    :  2.039631074550157
Win rate = 0.6556498254892255
-------------Round number:  37  -------------
Average Test Accuracy          :  0.29340196537201685
Average Global Trainning Accuracy:  0.29041896140382173
Average Global Trainning Loss    :  2.0327658504091506
Win rate = 0.6587452769301143
-------------Round number:  38  -------------
Average Test Accuracy          :  0.29794772377832746
Average Global Trainning Accuracy:  0.2945439140226092
Average Global Trainning Loss    :  2.026782056177395
Win rate = 0.6611044620869859
-------------R

...

Win rate = 0.9976188835070087
-------------Round number:  173  -------------
Average Test Accuracy          :  0.3969516678922388
Average Global Trainning Accuracy:  0.39410020290307474
Average Global Trainning Loss    :  1.7586052643314232
Win rate = 0.9976251496030428
-------------Round number:  174  -------------
Average Test Accuracy          :  0.3969516678922388
Average Global Trainning Accuracy:  0.3942562821913532
Average Global Trainning Loss    :  1.7580365155856317
Win rate = 0.99762828265106
-------------Round number:  175  -------------
Average Test Accuracy          :  0.3972190654455512
Average Global Trainning Accuracy:  0.39447925260317956
Average Global Trainning Loss    :  1.7574686378180115
Win rate = 0.9976408148431283
-------------Round number:  176  -------------
Average Test Accuracy          :  0.39802125810548833
Average Global Trainning Accuracy:  0.39508127271511073
Average Global Trainning Loss    :  1.7569016310285626
Win rate = 0.9977912011479488
--------

Average Test Accuracy          :  0.40176482385186174
Average Global Trainning Accuracy:  0.39942919574572455
Average Global Trainning Loss    :  1.741567363542108
Win rate = 0.9986214588724787
-------------Round number:  208  -------------
Average Test Accuracy          :  0.40169797446353367
Average Global Trainning Accuracy:  0.3991616312515329
Average Global Trainning Loss    :  1.741137274521171
Win rate = 0.9986277249685128
-------------Round number:  209  -------------
Average Test Accuracy          :  0.40203222140517414
Average Global Trainning Accuracy:  0.39956297799282037
Average Global Trainning Loss    :  1.7407075338915026
Win rate = 0.9986339910645471
-------------Round number:  210  -------------
Average Test Accuracy          :  0.40216592018183034
Average Global Trainning Accuracy:  0.3996075720751856
Average Global Trainning Loss    :  1.740278490044371
Win rate = 0.9986371241125641
-------------Round number:  211  -------------
Average Test Accuracy          :  0.4

Win rate = 0.9989159653860855
-------------Round number:  242  -------------
Average Test Accuracy          :  0.4058426365398757
Average Global Trainning Accuracy:  0.40350955428214674
Average Global Trainning Loss    :  1.7273845291979755
Win rate = 0.9989190984341026
-------------Round number:  243  -------------
Average Test Accuracy          :  0.4066448291998128
Average Global Trainning Accuracy:  0.40428995072353896
Average Global Trainning Loss    :  1.727014537670851
Win rate = 0.9989159653860855
-------------Round number:  244  -------------
Average Test Accuracy          :  0.40631058225817235
Average Global Trainning Accuracy:  0.40404468327053
Average Global Trainning Loss    :  1.7266452429262638
Win rate = 0.9989096992900514
-------------Round number:  245  -------------
Average Test Accuracy          :  0.4059763353165319
Average Global Trainning Accuracy:  0.40371022765279047
Average Global Trainning Loss    :  1.7262766449642133
Win rate = 0.9989222314821197
---------

Average Test Accuracy          :  0.41025469616953003
Average Global Trainning Accuracy:  0.4069210015830899
Average Global Trainning Loss    :  1.715595665455194
Win rate = 0.9991948066596069
-------------Round number:  277  -------------
Average Test Accuracy          :  0.41012099739287383
Average Global Trainning Accuracy:  0.4068987045419073
Average Global Trainning Loss    :  1.715271661575509
Win rate = 0.9991854075155556
-------------Round number:  278  -------------
Average Test Accuracy          :  0.40992044922788956
Average Global Trainning Accuracy:  0.4067872193359941
Average Global Trainning Loss    :  1.7149527093692167
Win rate = 0.9991854075155556
-------------Round number:  279  -------------
Average Test Accuracy          :  0.4098535998395615
Average Global Trainning Accuracy:  0.4068318134183594
Average Global Trainning Loss    :  1.7146337571629244
Win rate = 0.9991791414195214
-------------Round number:  280  -------------
Average Test Accuracy          :  0.409

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     url                   : https://www.comet.com/abdulmomen96/sophia/e1aa7781b43d4ab59bb3593c3852e0cb
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     glob_acc [299]   : (0.0981349020656461, 0.4119259308777325)
COMET INFO:     loss [720]       : (1.676066517829895, 2.311276435852051)
COMET INFO:     train_acc [299]  : (0.09850832794488171, 0.4081696358893175)
COMET INFO:     train_loss [299] : (1.708485348056813, 2.304447458416018)
COMET INFO:   Uploads:
COMET INFO:     conda-environment-definition : 1
COMET INFO:     conda-info                   : 1
COMET INFO:     conda-specification          : 1
COMET INFO:     environment details          : 1
COMET INFO:     filen

In [3]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **sophiaOTA_params)
experiment.end()  

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/abdulmomen96/sophia/040473488c58419480769ccf060782bc



---------------Running time:------------ 0


COMET WARNING: Failed to log system metrics: [sys.ram,sys.cpu,sys.load]


Number of edges / total edges: 20  /  20
Total number of parameters: 421642
-------------Round number:  0  -------------


RuntimeError: Given groups=1, weight of size [32, 1, 3, 3], expected input[3026, 3, 32, 32] to have 1 channels, but got 3 channels instead

In [2]:
experiment = Experiment(
        api_key="q24VqIVkFNEOugLA3T0YFFFvE",
        project_name="sophia",
        workspace="abdulmomen96",
    )
main(experiment, **GD_params)
experiment.end()    

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/abdulmomen96/sophia/ab642d4157be40a78382c15d30779c31



---------------Running time:------------ 0
Number of edges / total edges: 20  /  20
-------------Round number:  0  -------------
Average Test Accuracy          :  0.10234641353031619
Average Global Trainning Accuracy:  0.10265557760485183
Average Global Trainning Loss    :  2.3045032010189748
-------------Round number:  1  -------------
Average Test Accuracy          :  0.13236178888963165
Average Global Trainning Accuracy:  0.13625721866708287
Average Global Trainning Loss    :  2.303483633971772
-------------Round number:  2  -------------
Average Test Accuracy          :  0.15956948993916706
Average Global Trainning Accuracy:  0.1611853107092689
Average Global Trainning Loss    :  2.3024987318557826
-------------Round number:  3  -------------
Average Test Accuracy          :  0.15763085767765225
Average Global Trainning Accuracy:  0.15953532966175388
Average Global Trainning Loss    :  2.301493623046222
-------------Round number:  4  -------------
Average Test Accuracy          :  

-------------Round number:  39  -------------
Average Test Accuracy          :  0.2526238384918778
Average Global Trainning Accuracy:  0.25251399139334213
Average Global Trainning Loss    :  2.1253611075497783
-------------Round number:  40  -------------
Average Test Accuracy          :  0.25362657931679927
Average Global Trainning Accuracy:  0.25182278311668044
Average Global Trainning Loss    :  2.115059526132132
-------------Round number:  41  -------------
Average Test Accuracy          :  0.25422822381175214
Average Global Trainning Accuracy:  0.2536288434524739
Average Global Trainning Loss    :  2.1054792888358715
-------------Round number:  42  -------------
Average Test Accuracy          :  0.2558326091316264
Average Global Trainning Accuracy:  0.25389640794666546
Average Global Trainning Loss    :  2.097260564616825
-------------Round number:  43  -------------
Average Test Accuracy          :  0.2580386389464536
Average Global Trainning Accuracy:  0.2560146268590158
Average

Average Test Accuracy          :  0.3360518751253426
Average Global Trainning Accuracy:  0.3342772414100649
Average Global Trainning Loss    :  1.8943061021427456
-------------Round number:  79  -------------
Average Test Accuracy          :  0.3434721572297613
Average Global Trainning Accuracy:  0.34009676915873266
Average Global Trainning Loss    :  1.8885991047737964
-------------Round number:  80  -------------
Average Test Accuracy          :  0.3448759943846514
Average Global Trainning Accuracy:  0.3425048496064572
Average Global Trainning Loss    :  1.8837066461905505
-------------Round number:  81  -------------
Average Test Accuracy          :  0.34574503643291665
Average Global Trainning Accuracy:  0.33978461058217574
Average Global Trainning Loss    :  1.877769362193137
-------------Round number:  82  -------------
Average Test Accuracy          :  0.3504244936158834
Average Global Trainning Accuracy:  0.34373118687150217
Average Global Trainning Loss    :  1.872355361880978

-------------Round number:  118  -------------
Average Test Accuracy          :  0.39795440871716026
Average Global Trainning Accuracy:  0.3847800396887333
Average Global Trainning Loss    :  1.7270643576222435
-------------Round number:  119  -------------
Average Test Accuracy          :  0.39093522294271005
Average Global Trainning Accuracy:  0.38069968115231106
Average Global Trainning Loss    :  1.7259937512542085
-------------Round number:  120  -------------
Average Test Accuracy          :  0.4022327695701584
Average Global Trainning Accuracy:  0.3894847153782693
Average Global Trainning Loss    :  1.7197077275970478
-------------Round number:  121  -------------
Average Test Accuracy          :  0.3994250952603784
Average Global Trainning Accuracy:  0.3878347343307543
Average Global Trainning Loss    :  1.7155353937657474
-------------Round number:  122  -------------
Average Test Accuracy          :  0.4049735944916104
Average Global Trainning Accuracy:  0.3949028963856496
Av

...

Average Test Accuracy          :  0.3095795173474163
Average Global Trainning Accuracy:  0.3074539008673549
Average Global Trainning Loss    :  1.970534634272782
-------------Round number:  66  -------------
Average Test Accuracy          :  0.3116518483855873
Average Global Trainning Accuracy:  0.30975049610916633
Average Global Trainning Loss    :  1.965155995674374
-------------Round number:  67  -------------
Average Test Accuracy          :  0.3085767765224948
Average Global Trainning Accuracy:  0.30769916832036387
Average Global Trainning Loss    :  1.959216098742447
-------------Round number:  68  -------------
Average Test Accuracy          :  0.31506116719032023
Average Global Trainning Accuracy:  0.3130504582041963
Average Global Trainning Loss    :  1.9530058501861802
-------------Round number:  69  -------------
Average Test Accuracy          :  0.3125208904338525
Average Global Trainning Accuracy:  0.3116903386920556
Average Global Trainning Loss    :  1.946993836261678
--

-------------Round number:  105  -------------
Average Test Accuracy          :  0.37382177953071727
Average Global Trainning Accuracy:  0.36509175232446656
Average Global Trainning Loss    :  1.7779183342995384
-------------Round number:  106  -------------
Average Test Accuracy          :  0.3783675379370279
Average Global Trainning Accuracy:  0.36781199134874804
Average Global Trainning Loss    :  1.7730453856273272
-------------Round number:  107  -------------
Average Test Accuracy          :  0.37923657998529314
Average Global Trainning Accuracy:  0.36872617003723607
Average Global Trainning Loss    :  1.768754772960378
-------------Round number:  108  -------------
Average Test Accuracy          :  0.37883548365532455
Average Global Trainning Accuracy:  0.36941737831389776
Average Global Trainning Loss    :  1.766790368793061
-------------Round number:  109  -------------
Average Test Accuracy          :  0.37883548365532455
Average Global Trainning Accuracy:  0.3693281901491672

-------------Round number:  144  -------------
Average Test Accuracy          :  0.42716759141653854
Average Global Trainning Accuracy:  0.4161073825503356
Average Global Trainning Loss    :  1.6422613937880444
-------------Round number:  145  -------------
Average Test Accuracy          :  0.4262316999799452
Average Global Trainning Accuracy:  0.4147472630381948
Average Global Trainning Loss    :  1.6332328340654194
-------------Round number:  146  -------------
Average Test Accuracy          :  0.4305769102212715
Average Global Trainning Accuracy:  0.4176012843095721
Average Global Trainning Loss    :  1.6299649239670895
-------------Round number:  147  -------------
Average Test Accuracy          :  0.4321144461528177
Average Global Trainning Accuracy:  0.41882762157461706
Average Global Trainning Loss    :  1.6256356398693392
-------------Round number:  148  -------------
Average Test Accuracy          :  0.42883882612474095
Average Global Trainning Accuracy:  0.4184931659568775
Av

-------------Round number:  183  -------------
Average Test Accuracy          :  0.4593221472023531
Average Global Trainning Accuracy:  0.4499096969832103
Average Global Trainning Loss    :  1.5444268545564004
-------------Round number:  184  -------------
Average Test Accuracy          :  0.4554448826793235
Average Global Trainning Accuracy:  0.4462975763116234
Average Global Trainning Loss    :  1.5391093586256104
-------------Round number:  185  -------------
Average Test Accuracy          :  0.4562470753392606
Average Global Trainning Accuracy:  0.4485495774710696
Average Global Trainning Loss    :  1.5372543493165958
-------------Round number:  186  -------------
Average Test Accuracy          :  0.456514472892573
Average Global Trainning Accuracy:  0.4472117550001115
Average Global Trainning Loss    :  1.5403867352114875
-------------Round number:  187  -------------
Average Test Accuracy          :  0.4561133765626045
Average Global Trainning Accuracy:  0.44783607215322524
Avera

KeyboardInterrupt: 

In [ ]:
'''
Human Activity
Sophia
-------------Round number:  40  -------------
Average Test Accuracy          :  0.8502321981424149
Average Global Trainning Accuracy:  0.8495139338950097
Average Global Trainning Loss    :  1.0823885845349968
Clipping rate = 0.576000928907216

"learning_rate": 0.0001,
    "alpha": (0.9, 0.95),
    "eta": 1.0 * 50,
    "L": 0.001,
    "rho": 20,


GD
-------------Round number:  40  -------------
Average Global Accuracy          :  0.20936532507739938
Average Global Trainning Accuracy:  0.20764744005184704
Average Global Trainning Loss    :  1.7220779072828905


'''

'''
MNIST MLP
Sophia 
-------------Round number:  39  -------------
Average Global Accuracy          :  0.8934867045115028
Average Global Trainning Accuracy:  0.8913894080996885
Average Global Trainning Loss    :  0.3557637022975078

GD
-------------Round number:  40  -------------
Average Global Accuracy          :  0.5679713175978488
Average Global Trainning Accuracy:  0.5671775700934579
Average Global Trainning Loss    :  1.9333872663551401

'''

In [ ]:
from utils.plot_utils import get_training_data_value, simple_read_data

In [ ]:


rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Mnist_Sophia_0.01_(0.9, 0.95)_20.0_0.1_32u_64b_1_0_25db")
rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Mnist_GD_0.01_0.03_20.0_0.001_30u_64b_1_0")

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop[:80], label='FedSophia-25dB', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
#plt.plot(x, rs_glob_acc_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop[:80], label='FedSophia-25dB', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
#plt.plot(x, rs_train_loss_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:


rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Fashion_Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.1_20u_64b_1_0")
#rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Fashion_Mnist_GD_0.01_0.03_20.0_0.001_20u_64b_1_0")


# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_glob_acc_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_train_loss_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:


rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Mnist_Sophia_0.005_(0.9, 0.95)_20.0_0.1_32u_64b_1_0_CNN_IID")
#rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Mnist_GD_0.01_0.03_20.0_0.001_30u_64b_1_0_CNN_IID")

b = rs_train_loss_gd

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_glob_acc_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_train_loss_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:


rs_train_acc_sop, rs_train_loss_sop, rs_glob_acc_sop = simple_read_data("Mnist_Sophia_0.005_(0.9, 0.95)_20.0_0.1_32u_64b_1_0_CNN_NIID")
#rs_train_acc_done, rs_train_loss_done, rs_glob_acc_done = simple_read_data("Mnist_DONE_1_0.003_1.0_0.001_32u_0b_40_0")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_gd, rs_train_loss_gd, rs_glob_acc_gd = simple_read_data("Mnist_GD_0.01_0.03_20.0_0.001_30u_64b_1_0_CNN_NIID")

a = rs_train_loss_gd

# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41
print(rs_glob_acc_sop[:80].shape)
# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_glob_acc_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_glob_acc_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop[:80], label='FedSophia', linestyle='--', marker='o', markevery=4)
#plt.plot(x_DONE, rs_train_loss_done, label='DONE', linestyle='--', marker='s', markevery=4)
plt.plot(x, rs_train_loss_gd, label='FedAVG', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:

# Sopgia with SNRs

from utils.plot_utils import get_training_data_value, simple_read_data
import matplotlib.pyplot as plt
import numpy as np
rs_train_acc_sop25, rs_train_loss_sop25, rs_glob_acc_sop25 = simple_read_data("Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.001_32u_64b_1_0_25dB")
rs_train_acc_sop30, rs_train_loss_sop30, rs_glob_acc_sop30 = simple_read_data("Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.001_32u_64b_1_0_30dB")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_sop35, rs_train_loss_sop35, rs_glob_acc_sop35 = simple_read_data("Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.001_32u_64b_1_0_35dB")


# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41

# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop25[:80], label='SNR=25dB', linestyle='--', marker='o', markevery=4)
plt.plot(x[:80], rs_glob_acc_sop30[:80], label='SNR=30dB', linestyle='--', marker='s', markevery=4)
plt.plot(x[:80], rs_glob_acc_sop35[:80], label='SNR=35dB', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop25[:80], label='SNR=25dB', linestyle='--', marker='o', markevery=4)
plt.plot(x[:80], rs_train_loss_sop30[:80], label='SNR=30dB', linestyle='--', marker='s', markevery=4)
plt.plot(x[:80], rs_train_loss_sop35[:80], label='SNR=35dB', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:

# Sopgia with SNRs

from utils.plot_utils import get_training_data_value, simple_read_data
import matplotlib.pyplot as plt
import numpy as np
rs_train_acc_sop25, rs_train_loss_sop25, rs_glob_acc_sop25 = simple_read_data("Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.001_32u_64b_1_0_fix_5dB")
rs_train_acc_sop30, rs_train_loss_sop30, rs_glob_acc_sop30 = simple_read_data("Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.001_32u_64b_1_0_fix_25dB")
#("Mnist_DONE_1_0.008_1.0_0.001_32u_0b_40_0")
rs_train_acc_sop35, rs_train_loss_sop35, rs_glob_acc_sop35 = simple_read_data("Mnist_Sophia_0.001_(0.9, 0.95)_20.0_0.001_32u_64b_1_0_fix_35dB")


# Create a figure with two subplots
plt.figure(figsize=(10, 5))
x = np.arange(0, 82)
x_DONE = np.arange(0, 2 * 41, 2)
xlim = 41

# Subplot 1: Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(x[:80], rs_glob_acc_sop25[:80], label='SNR=5dB', linestyle='--', marker='o', markevery=4)
plt.plot(x[:80], rs_glob_acc_sop30[:80], label='SNR=25dB', linestyle='--', marker='s', markevery=4)
plt.plot(x[:80], rs_glob_acc_sop35[:80], label='SNR=35dB', linestyle='--', marker='D', markevery=4)

plt.title('Test Accuracy')
plt.xlabel('Communication Rounds')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Training Loss
plt.subplot(1, 2, 2)
plt.plot(x[:80], rs_train_loss_sop25[:80], label='SNR=5dB', linestyle='--', marker='o', markevery=4)
plt.plot(x[:80], rs_train_loss_sop30[:80], label='SNR=25dB', linestyle='--', marker='s', markevery=4)
plt.plot(x[:80], rs_train_loss_sop35[:80], label='SNR=35dB', linestyle='--', marker='D', markevery=4)

plt.title('Training Loss')
plt.xlabel('Communication Rounds')
plt.ylabel('Loss')
plt.legend()

plt.savefig('Mnist_MLP_.eps', format='eps')

plt.tight_layout()
plt.show()


In [ ]:
a - b

In [ ]:
#1.027

np.mean(np.equal(np.ones((20, 1), np.float32), 0.0))

In [ ]:
model = DNN( input_dim = 123, output_dim = 2)


In [ ]:

        #print(grads[i].sign())
#print(f"Clipping rate = {1 - (winrate / param_count)}")
x = sum(p.numel() for p in model.parameters() if p.requires_grad)
x 

In [ ]:
"""
Mnist MLP settings
sophia_params = {
    "dataset": "Mnist",
    "algorithm": "Sophia",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": (0.90, 0.95),
    "eta": 1.0 * 20,
    "L": 0.1,
    "rho": 20,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "Sophia",
    "numedges": 32,
    "times": 1,
    "commet": False,
    "gpu": 0
}



DONE_params = {
    "dataset": "Mnist",
    "algorithm": "DONE",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 1,
    "alpha": 0.003,
    "eta": 1.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 41,
    "local_epochs": 40,
    "optimizer": "DONE",
    "numedges": 32,
    "times": 1,
    "commet": True,
    "gpu": 0
}

GD_params = {
    "dataset": "Mnist",
    "algorithm": "GD",
    "model": "MLP",
    "batch_size": 64,
    "learning_rate": 0.01,
    "alpha": 0.03,
    "eta": 20.0,
    "L": 1e-3,
    "rho": 0.01,
    "num_glob_iters": 2 * 41,
    "local_epochs": 1,
    "optimizer": "DONE",
    "numedges": 30,
    "times": 1,
    "commet": False,
    "gpu": 0
}







"""

In [ ]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 2, 1)
        self.conv2 = nn.Conv2d(16, 32, 2, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(18432, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = nn.ReLU()(x)
        x = nn.MaxPool2d(2, 1)(x)
        x = self.dropout1(x)
        x = self.conv2(x)
        x = nn.ReLU()(x)
        x = nn.MaxPool2d(2, 1)(x)
        x = self.dropout2(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = nn.ReLU()(x)
        x = self.fc2(x)
        output = x#F.log_softmax(x, dim=1)
        return output
    


In [ ]:
model = Net()

In [ ]:
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of parameters: {param_count}")

In [ ]:
class CNN28(nn.Module):
    def __init__(self):
        super(CNN28, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # Adjusted input size based on the dimensions after pooling
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(nn.ReLU()(self.conv1(x)))
        x = self.pool(nn.ReLU()(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = nn.ReLU()(self.fc1(x))
        x = self.fc2(x)
        return x

# Create an instance of the model
model = CNN28()

In [ ]:
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of parameters: {param_count}")